# 🚀 Googleドライブ動画 超高速GPU文字起こし（Google Colab T4 GPU用）
1. 上部メニューの **「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」** が選ばれていることを確認してください。
2. 下の再生ボタン（▶）を **1回押すだけ** で、Googleドライブから動画を直接高速取得し、GPUで最高精度の文字起こしを行い完了後に自動で `transcript_cas6.txt` がパソコンにダウンロードされます（所要時間：約3〜5分）。

In [ ]:
# @title ▶ ここを1回押すだけで文字起こし＆自動ダウンロードまで完了します
FILE_ID_OR_URL = "https://drive.google.com/file/d/1SUdCJ7T3LU3MzW_PuSwq-1R1URwQkieg/view?usp=sharing" # @param {type:"string"}
OUTPUT_FILENAME = "transcript_cas6.txt" # @param {type:"string"}

import os, re, time, subprocess

print("1/4 📦 必要なライブラリ (gdown / faster-whisper) をインストール中...")
subprocess.run(["pip", "install", "-q", "-U", "gdown", "faster-whisper"], check=True)

m = re.search(r"(?:/d/|id=)([a-zA-Z0-9_-]{20,})", FILE_ID_OR_URL)
file_id = m.group(1) if m else FILE_ID_OR_URL.strip()

print(f"2/4 ⚡ Googleドライブから動画をクラウド間高速ダウンロード中 (ID: {file_id})...")
import gdown
video_path = "input_video.mp4"
gdown.download(id=file_id, output=video_path, quiet=False)

print("3/4 🎵 音声データ (16kHz モノラル) を高速抽出中...")
audio_path = "audio_16k.wav"
subprocess.run([
    "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
    "-i", video_path, "-vn", "-ac", "1", "-ar", "16000", audio_path
], check=True)

print("4/4 🚀 GPU (large-v3-turbo) で爆速文字起こし中...")
from faster_whisper import WhisperModel, BatchedInferencePipeline

t0 = time.time()
model = WhisperModel("deepdml/faster-whisper-large-v3-turbo-ct2", device="cuda", compute_type="float16")
batched_model = BatchedInferencePipeline(model=model)
segments, info = batched_model.transcribe(
    audio_path,
    language="ja",
    batch_size=16,
    vad_filter=True
)

lines = []
for s in segments:
    st = int(s.start)
    hh, rem = divmod(st, 3600)
    mm, ss = divmod(rem, 60)
    ts = f"{hh:02d}:{mm:02d}:{ss:02d}" if hh > 0 else f"{mm:02d}:{ss:02d}"
    line = f"[{ts}] {s.text.strip()}"
    lines.append(line)
    if len(lines) % 100 == 0:
        print(f"  ... {ts} まで完了 ({len(lines)} 行)")

with open(OUTPUT_FILENAME, "w", encoding="utf-8-sig") as f:
    f.write("\n".join(lines) + "\n")

print(f"✅ 文字起こし完了！ 所要時間: {time.time() - t0:.1f}秒 / 合計 {len(lines)} 行")
from google.colab import files
files.download(OUTPUT_FILENAME)
